In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.stats import sigma_clipped_stats

from sklearn.cluster import DBSCAN

from Kakapo.photometry import forced_photometry

%matplotlib widget

In [9]:
def initial_filter(df):
    df = df[(df.fwhm > 0.8) & (df.snr >= 0.5) & 
            (df.psfdiff <= 2) & (df.poisson_thresh >= 0.5) & 
            (abs(df.correlation) >= 0.05)]
    
    return df

def _grouping(corr: pd.DataFrame, f_dist: int = 48) -> pd.DataFrame | None:
    """
    Group detections with DBSCAN in O(N log N) time, using only C/Fortran
    code paths from scikit-learn (no Python callback per point pair).
    """
    if corr.empty:
        return None

    # Scale the frame axis so that `eps` of 1.25 encloses ±f_dist frames.
    data = corr[['xcentroid', 'ycentroid', 'frame']].values.astype(np.float32)
    data[:, 2] *= 1.5 / f_dist

    db = DBSCAN(eps=1.5,
                min_samples=5,
                metric='euclidean',           # now fully compiled
                algorithm='auto',        # fastest for 3‑D Euclidean
                n_jobs=1)                     # keep it serial – you already parallelise at a higher level
    labels = db.fit_predict(data)

    corr = corr.assign(cluster=labels)
    corr = corr[corr.cluster != -1]           # drop noise points

    return corr if not corr.empty else None

# def mask_detections(correlation, psfdiff, fwhm, snr, 
#                     roundness, poisson_thresh, xstd, ystd):
    
#     mask =  (correlation >= self.corrlim) & (psfdiff <= self.difflim) & \
#             (fwhm <= self.fwhmlim) & (fwhm >= 0.9) & \
#             (snr >= self.snrlim) & (snr < 10000) & (abs(roundness) <= self.roundness) & \
#             (poisson_thresh >= self.poiss_val) & \
#             (xstd <= self.dist_cut) & (ystd <= self.dist_cut)
    
    
#     return mask

In [10]:
file = '/Users/zgl12/Modules/Kakapo/Data/csv_files/c3/c3_t205922648.csv'
df = pd.read_csv(file)

In [11]:
df_new = df[(df['fwhm'] >= 0.8) & (df['fwhm'] <= 5) & 
   (df['roundness'] <= 0.95) & (abs(df['correlation'])>= 0.2) & 
   (abs(df['snr'])>= 3) & (abs(df['snr'])<= 1e4) & (df['psfdiff'] <= 1.2) & 
   (df['psfdiff'] <= 1.2) & (df['poisson_thresh'] >= 1)]

In [12]:
# plt.figure()
# plt.scatter(df.xcentroid.values, df.ycentroid.values, c = df.frame.values)
# plt.xlabel(r'$x$')
# plt.ylabel(r'$y$')
# plt.show()

# plt.figure()
# plt.scatter(df.fwhm.values, df.roundness.values, c = df.frame.values)
# plt.xlabel(r'FWHM')
# plt.ylabel(r'Roundness')
# plt.show()

# plt.figure()
# plt.scatter(df.snr.values, df.correlation.values, c = df.frame.values)
# plt.xlabel(r'SNR')
# plt.ylabel(r'Correlation')
# plt.show()

# plt.figure()
# plt.scatter(df.psfdiff.values, df.poisson_thresh.values, c = df.frame.values)
# plt.xlabel(r'PSF-Diff.')
# plt.ylabel(r'Poisson Threshold')
# plt.show()

In [13]:
df_1 = initial_filter(df)

df_1 = _grouping(df_1, f_dist = 48)

In [14]:
df_1

,xcentroid,ycentroid,fwhm,roundness,pa,max_value,flux,mag,snr,flux_err,...,n_detections,poisson_thresh,ref_flux,campaign,target_id,ra,dec,filename,ref_frame,cluster
260,7.449393,6.442264,1.117837,0.642882,97.476346,21.619372,105.711449,-5.060305,1.848658,57.182795,...,1,0.533930,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,0
270,6.968819,7.147297,1.203694,0.257459,105.044898,24.426942,116.016630,-5.161301,1.230878,94.255215,...,1,0.519521,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,0
271,6.951694,7.101232,1.242587,0.277224,104.291474,24.639601,117.787096,-5.177744,1.410986,83.478578,...,1,0.524051,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,0
272,6.926905,7.144274,1.241937,0.302403,103.284029,24.515847,117.020611,-5.170656,1.529148,76.526685,...,1,0.521428,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,0
274,7.143648,7.298206,0.994854,0.330019,123.050799,23.364979,110.819425,-5.111540,1.243358,89.129137,...,1,0.537114,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
933,7.452678,6.217127,1.275800,0.359101,102.574531,37.597065,195.873338,-5.729938,4.042932,48.448337,...,1,0.799673,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,0
934,7.492137,6.193089,1.259153,0.311644,102.450236,34.352394,176.577142,-5.617336,2.332826,75.692368,...,1,0.730491,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,0
935,7.483612,6.179442,1.269569,0.320416,103.171803,37.773973,194.617151,-5.722953,1.776746,109.535730,...,1,0.803221,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,0
936,7.441055,6.079064,1.327072,0.327415,104.223953,41.182242,221.386338,-5.862877,2.470000,89.630085,...,1,0.875706,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,0


In [15]:
for cluster in np.unique(df_1.cluster.values):
    
    temp_df = df_1[df_1.cluster == cluster]
    print(cluster)
    print(len(temp_df), temp_df.frame.min(), temp_df.frame.max())
    x, _, xstd = sigma_clipped_stats(temp_df.xcentroid.values, sigma = 3)
    y, _, ystd = sigma_clipped_stats(temp_df.ycentroid.values, sigma = 3)
    
    print(f"{x:.2f} +/- {xstd:.2f}")
    print(f"{y:.2f} +/- {ystd:.2f}")
    
    # x, _, xstd = sigma_clipped_stats(temp_df.xcentroid.values, sigma = 3)
    # y, _, ystd = sigma_clipped_stats(temp_df.ycentroid.values, sigma = 3)
    correlation, _, _ = sigma_clipped_stats(temp_df.correlation.values, sigma = 3)
    psfdiff, _, _  = sigma_clipped_stats(temp_df.psfdiff.values, sigma = 3)
    snr, _, _  = sigma_clipped_stats(temp_df.snr.values, sigma = 3)
    fwhm, _, _  = sigma_clipped_stats(temp_df.fwhm.values, sigma = 3)
    roundness, _, _  = sigma_clipped_stats(temp_df.roundness.values, sigma = 3)
    poisson_thresh, _, _  = sigma_clipped_stats(temp_df.poisson_thresh.values, sigma = 3)

    print('Corr.', correlation)
    print('PSF Diff.', psfdiff)
    print('SNR', snr)
    print('FWHM', fwhm)
    print('Roundness', roundness)
    print('Poiss.', poisson_thresh)
    print()


0
646 2635.0 3385.0
7.31 +/- 0.17
6.45 +/- 0.47
Corr. 0.8970450184762692
PSF Diff. 0.521339010771436
SNR 3.1179417224692325
FWHM 1.3030748723279215
Roundness 0.3281156654445752
Poiss. 0.8649262022217651



In [ ]:


# mask_detections(correlation, psfdiff, fwhm, snr, 
#                     roundness, poisson_thresh, xstd, ystd)

In [ ]:
diff = np.load('/Users/zgl12/Modules/Kakapo/Data/difference_arrays/c3/diff_c3_t205922648.npy')

In [ ]:
fluxes = forced_photometry(diff, 6.36, 6.35, None)

In [ ]:
plt.figure()
plt.plot(fluxes)
plt.axvline(2695.0, color = 'r')
plt.axvline(3385.0, color = 'r')
plt.show()